# OOD Detection Demo — MSP · KNN · ViM

This notebook scores **three randomly drawn images** with three out-of-distribution detection
methods, all sharing one WideResNet-40-2 trained on CIFAR-10.

| | Dataset | Role |
|---|---|---|
| ID | CIFAR-10 | what the model was trained to recognise |
| OOD 1 | CIFAR-100 | **near-OOD** — natural 32×32 images, many classes semantically close to CIFAR-10 |
| OOD 2 | MNIST | **far-OOD** — handwritten digits, grayscale |

| Method | Score | Direction |
|---|---|---|
| **MSP** | `max(softmax(logits))` — the model's confidence | high = ID |
| **KNN** | distance to the K-th neighbour in the CIFAR-10 train feature bank | high = OOD |
| **ViM** | virtual logit matching: residual outside the principal subspace, minus energy | high = OOD |

> **Note on direction:** MSP runs opposite to the other two. High MSP means *looks like ID*,
> while high KNN and high ViM mean *looks like OOD*.

---

## How to run

1. Put `ood_artifacts.npz` next to this notebook.
   On Colab: click the folder icon in the left sidebar, then `Upload`.
2. `Runtime` → `Run all`.

Takes about **10 seconds**. No GPU required, no dataset download.

## Why it is this fast

The companion notebook `OOD_MSP_KNN_ViM_demo.ipynb` downloads 335 MB of datasets and makes a full
pass over 50,000 training images to build the KNN feature bank and fit ViM. That work has already
been done; its output is packaged in `ood_artifacts.npz`:

| In the archive | Used by | Size |
|---|---|---|
| `feature_bank`, 50,000 × 128 | KNN | ~25 MB |
| `principal_subspace` 128 × 64, plus `alpha` | ViM | 33 KB |
| — | MSP | **nothing** — the model is enough |
| 200 images per dataset | drawing demo images | ~1.4 MB |

The only bulky component is KNN's feature bank, and that is inherent to the method: KNN is
non-parametric, so it **must** carry its reference data. MSP costs nothing because it only reads a
softmax. ViM learned exactly one 128 × 64 projection and a single scalar.

**This notebook still runs the real model on the three images.** Nothing is read from a table of
precomputed answers — only the *trained* state is loaded; the scoring happens live. Cell 2 prints
the provenance of the archive, including a SHA-256 digest of the feature bank, which can be
compared against what the training notebook printed when it wrote the file.

The one thing still downloaded is the model checkpoint itself (8.6 MB, a few seconds).

In [ ]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES
# ============================================================

import subprocess
import sys

try:
    import pytorch_ood
except ModuleNotFoundError:
    # sys.executable installs into the exact Python running this notebook
    print("Installing pytorch-ood ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pytorch-ood"],
        check=True,
    )
    import pytorch_ood

print("Python      :", sys.version.split()[0])
print("pytorch-ood :", pytorch_ood.__version__)

## 1. Load the artifact

`SEED = None` draws a different set of three images each time Cell 4 runs. Set it to an integer to
reproduce one specific draw.

The provenance block printed below identifies the training run that produced this archive. The
`feature SHA-256` line should match the digest printed by the training notebook when it exported
the file.

In [ ]:
# ============================================================
# CELL 2: IMPORTS + LOAD THE ARTIFACT
# ============================================================

import os

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from PIL import Image
from torchvision import transforms

ARTIFACT_PATH = "ood_artifacts.npz"

# None = draw different images every run. Set an integer to reproduce one draw.
SEED = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not os.path.exists(ARTIFACT_PATH):
    raise FileNotFoundError(
        f"Could not find {ARTIFACT_PATH}.\n"
        "Place it in the same folder as this notebook.\n"
        "On Colab: click the folder icon in the left sidebar, then Upload."
    )

art = np.load(ARTIFACT_PATH, allow_pickle=False)

# Configuration recorded when the artifact was created
MODEL_ID        = str(art["model_id"])
K               = int(art["knn_k"])
VIM_D           = int(art["vim_d"])
CIFAR10_CLASSES = [str(c) for c in art["cifar10_classes"]]

print("Device:", DEVICE)
print()
print("=" * 72)
print("ARTIFACT PROVENANCE")
print("=" * 72)
print(f"  file               : {ARTIFACT_PATH}  ({os.path.getsize(ARTIFACT_PATH) / 1e6:.1f} MB)")
print(f"  model              : {MODEL_ID}")
print(f"  training images    : {int(art['n_train']):,}")
print(f"  feature bank shape : {art['feature_bank'].shape}  {art['feature_bank'].dtype}")
print(f"  KNN K              : {K}")
print(f"  ViM d              : {VIM_D}")
print(f"  ViM alpha          : {float(art['vim_alpha']):.6f}")
print(f"  trained on device  : {str(art['device_used'])}")
print(f"  created at (UTC)   : {str(art['created_at'])}")
print(f"  feature SHA-256    : {str(art['feature_sha256'])}")
print("=" * 72)

## 2. Verify the archive, then rebuild the model and detectors

**Integrity check.** The cell recomputes a SHA-256 digest over the feature bank it just read and
compares it against the digest recorded at export time. A match means the file is byte-for-byte
the one the training run produced.

**KNN** needs only the feature bank, loaded straight from the archive.

**ViM** would normally have to `fit` on 50,000 training images to derive its principal subspace and
alpha. Both are already in the archive, so they are assigned directly and **nothing is refitted**.
Order matters: assign the tensors **before** calling `.to(device)`, because `.to()` moves the
attributes that exist at that moment — assigning afterwards would leave them on the CPU and cause
a device mismatch during scoring.

The cell also cross-checks `vim.u`. That value is recomputed by `ViM.__init__` from the model's
final-layer weights (`u = -W⁺b`), so if it matches the stored copy, the model just loaded is
provably the same one used to build the archive.

In [ ]:
# ============================================================
# CELL 3: VERIFY, THEN REBUILD MODEL + KNN + ViM (NO REFITTING)
# ============================================================

import hashlib

from pytorch_ood.detector import ViM
from pytorch_ood.model import load_model, load_transform

# ------------------------------------------------------------
# Integrity: does the feature bank still hash to the recorded digest?
# ------------------------------------------------------------
digest = hashlib.sha256(art["feature_bank"].tobytes()).hexdigest()
expected = str(art["feature_sha256"])
assert digest == expected, (
    "SHA-256 mismatch -- the feature bank does not match the recorded digest.\n"
    f"  expected: {expected}\n  computed: {digest}"
)
print("SHA-256 verified: feature bank matches the training run.\n")

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------
model = load_model(MODEL_ID).to(DEVICE)
model.eval()

base_transform = load_transform(MODEL_ID)

# MNIST: 28x28 grayscale -> 32x32 -> 3 channels -> the CIFAR-10 base_transform itself
mnist_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(num_output_channels=3),
    base_transform,
])

# ------------------------------------------------------------
# KNN: load the feature bank
# ------------------------------------------------------------
feature_bank = torch.from_numpy(art["feature_bank"]).float().to(DEVICE)
label_bank   = torch.from_numpy(art["label_bank"]).long()

# ------------------------------------------------------------
# ViM: assign the already-fitted parameters
# ------------------------------------------------------------
vim = ViM(model.features, d=VIM_D, w=model.fc.weight, b=model.fc.bias)
vim.principal_subspace = torch.from_numpy(art["vim_principal_subspace"]).float()
vim.alpha = float(art["vim_alpha"])
vim = vim.to(DEVICE)        # .to() last, so every tensor lands on the same device

# ------------------------------------------------------------
# Cross-check that this is the same model the archive was built from
# ------------------------------------------------------------
stored_u = torch.from_numpy(art["vim_u"]).float()
assert torch.allclose(vim.u.cpu(), stored_u, atol=1e-4), \
    "vim.u differs -- the loaded model is NOT the one used to build the artifact!"

_x = torch.zeros(1, 3, 32, 32, device=DEVICE)
with torch.no_grad():
    assert model.features(_x).shape[1] == feature_bank.shape[1], \
        "Feature dimensionality does not match the feature bank!"
    assert torch.allclose(model.fc(model.features(_x)), model(_x), atol=1e-5), \
        "fc(features(x)) != model(x)"

print("Model             :", type(model).__name__)
print("Feature bank      :", tuple(feature_bank.shape), "on", feature_bank.device)
print("ViM alpha         :", f"{vim.alpha:.6f}")
print("ViM subspace      :", tuple(vim.principal_subspace.shape))
print("\n-> All integrity checks passed. Ready to score.")

## 3. Scoring function

One function returns all three scores for one image. The model runs `features(x)` **once**, and
that single feature feeds all three methods — so the three scores necessarily describe the same
representation of the image:

- **MSP** — `fc(feature)` gives logits, then `max(softmax(logits))`
- **KNN** — `cdist` from the feature to the bank, distance to the `K`-th neighbour
- **ViM** — `vim.predict_features(feature)`

In [ ]:
# ============================================================
# CELL 4: SCORE ONE IMAGE WITH ALL THREE METHODS
# ============================================================

@torch.no_grad()
def score_all_methods(image_tensor):
    """image_tensor: a preprocessed image of shape (3, 32, 32)."""
    model.eval()

    x = image_tensor.unsqueeze(0).to(DEVICE)
    feat = model.features(x)                 # (1, 128) -- computed ONCE

    # --- MSP ---
    logits = model.fc(feat)
    msp = F.softmax(logits, dim=1).max(dim=1).values.item()
    pred = logits.argmax(dim=1).item()

    # --- KNN ---
    dists = torch.cdist(feat, feature_bank).squeeze(0)
    knn_d, knn_i = torch.topk(dists, k=K, largest=False)

    # --- ViM ---
    vim_score = vim.predict_features(feat).item()

    return {
        "msp":       msp,
        "knn":       knn_d[-1].item(),
        "vim":       vim_score,
        "pred":      CIFAR10_CLASSES[pred],
        "neighbors": [CIFAR10_CLASSES[c] for c in label_bank[knn_i.cpu()].tolist()],
    }


print("score_all_methods() defined.")

## 4. Results — three random images

**Rerun this cell to see different draws.** Images come from the pool of 200 per dataset bundled
in the archive. The `index` shown is the position in the original test set, so any draw can be
traced back.

In [ ]:
# ============================================================
# CELL 5: DRAW THREE RANDOM IMAGES AND SCORE THEM
# Rerun this cell for a different draw.
# ============================================================

rng = np.random.default_rng(SEED)

# (display name, ID/OOD, image pool, original indices, matching transform)
pools = [
    ("CIFAR-10",  "ID",  art["pool_cifar10"],  art["pool_cifar10_idx"],  base_transform),
    ("CIFAR-100", "OOD", art["pool_cifar100"], art["pool_cifar100_idx"], base_transform),
    ("MNIST",     "OOD", art["pool_mnist"],    art["pool_mnist_idx"],    mnist_transform),
]

results = []
for name, kind, pool, orig_idx, tf in pools:
    j = int(rng.integers(len(pool)))
    arr = pool[j]                      # uint8: (32,32,3) for CIFAR, (28,28) for MNIST

    img = Image.fromarray(arr)         # mode "RGB" or "L" depending on shape
    r = score_all_methods(tf(img))

    r.update(
        name=name,
        kind=kind,
        index=int(orig_idx[j]),        # position in the original test set
        # .convert("RGB"): a mode "L" image passed to imshow would be drawn
        # with the default colormap instead of grayscale
        image=img.convert("RGB"),
    )
    results.append(r)

# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------
W = 20
print("=" * (12 + 3 * W))
print("THREE METHODS ON THREE RANDOM IMAGES")
print("=" * (12 + 3 * W))
print(f"{'':<12}" + "".join(f"{r['name'] + ' (' + r['kind'] + ')':>{W}}" for r in results))
print(f"{'index':<12}" + "".join(f"{'#' + str(r['index']):>{W}}" for r in results))
print("-" * (12 + 3 * W))
print(f"{'MSP':<12}" + "".join(f"{'Confidence: ' + format(r['msp'], '.2f'):>{W}}" for r in results))
print(f"{'KNN':<12}" + "".join(f"{'Distance: ' + format(r['knn'], '.2f'):>{W}}" for r in results))
print(f"{'ViM':<12}" + "".join(f"{'Score: ' + format(r['vim'], '.2f'):>{W}}" for r in results))
print("-" * (12 + 3 * W))
print(f"{'predicted':<12}" + "".join(f"{r['pred']:>{W}}" for r in results))
print("=" * (12 + 3 * W))

print("\nMSP high = looks ID    (max softmax probability)")
print(f"KNN high = looks OOD   (distance to the {K}th neighbour)")
print(f"ViM high = looks OOD   (virtual logit matching, d={VIM_D})")

print(f"\nClasses of the {K} nearest neighbours (first 10):")
for r in results:
    print(f"  {r['name']:<10} {', '.join(r['neighbors'][:10])}")

### Figure

In [ ]:
# ============================================================
# CELL 6: PLOT THE RESULTS
# ============================================================

BG    = "#1c1c1c"
INK   = "#e8e8e6"
MUTED = "#9a9a96"
LINE  = "#3a3a3a"
C_ID  = "#56b870"   # green - ID
C_OOD = "#e2724a"   # orange - OOD

MONO = "DejaVu Sans Mono"

fig = plt.figure(figsize=(11.5, 6.6), facecolor=BG)

gs = fig.add_gridspec(1, 3, left=0.22, right=0.97, top=0.86, bottom=0.50, wspace=0.26)
axes = [fig.add_subplot(gs[0, i]) for i in range(3)]

for ax, r in zip(axes, results):
    color = C_ID if r["kind"] == "ID" else C_OOD

    ax.imshow(r["image"], interpolation="nearest")
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ax.spines.values():
        side.set_color(color)
        side.set_linewidth(1.6)

    ax.set_title(f"{r['kind']} IMAGE", fontsize=12.5, color=color,
                 fontfamily=MONO, pad=22)
    ax.text(0.5, 1.04, f"{r['name']}  ·  #{r['index']}",
            transform=ax.transAxes, ha="center", va="bottom",
            fontsize=9, color=MUTED, fontfamily=MONO)

col_x = [ax.get_position().x0 for ax in axes]
col_w = axes[0].get_position().width

y = 0.385
for label, field_name, key in [("MSP", "Confidence:", "msp"),
                               ("KNN", "Distance:",   "knn"),
                               ("ViM", "Score:",      "vim")]:
    fig.text(0.05, y, label, fontsize=13.5, color=INK, fontfamily=MONO, va="center")

    for x, r in zip(col_x, results):
        color = C_ID if r["kind"] == "ID" else C_OOD
        fig.text(x, y, field_name, fontsize=11.5, color=MUTED,
                 fontfamily=MONO, va="center", ha="left")
        fig.text(x + col_w, y, f"{r[key]:.2f}", fontsize=11.5, color=color,
                 fontfamily=MONO, va="center", ha="right")

    y -= 0.088

fig.add_artist(plt.Line2D([0.04, 0.97], [0.45, 0.45], color=LINE, linewidth=1.0))

fig.text(0.04, 0.085,
         "MSP high = looks ID        KNN high = looks OOD        ViM high = looks OOD",
         fontsize=10, color=MUTED, fontfamily=MONO)
fig.text(0.04, 0.035,
         f"WideResNet-40-2 / CIFAR-10   ·   KNN K={K}   ·   ViM d={VIM_D}",
         fontsize=10, color=MUTED, fontfamily=MONO)

plt.show()

## 5. Averages across the whole pool

A single draw of three images is a sample of size one. Per-image scores overlap substantially
between the datasets, so **any individual draw can show a method ranking an OOD image as more
ID-like than the ID image** — that happens for roughly 5–18% of random pairs depending on the
method, which makes it fairly common to see at least one such inversion in the figure above.

The cell below scores all 600 bundled images so the overall trend can be read alongside the single
draw. Across the full pool the ordering comes out as expected for all three methods.

In [ ]:
# ============================================================
# CELL 7: SCORE THE ENTIRE BUNDLED POOL (600 IMAGES)
# ============================================================

@torch.no_grad()
def score_pool(pool, tf, batch=64):
    """Score every image in a pool. Returns an (N, 3) array: MSP, KNN, ViM."""
    x = torch.stack([tf(Image.fromarray(a)) for a in pool])
    out = []
    for i in range(0, len(x), batch):
        f = model.features(x[i:i + batch].to(DEVICE))
        msp = F.softmax(model.fc(f), dim=1).max(dim=1).values
        knn = torch.topk(torch.cdist(f, feature_bank), k=K, largest=False).values[:, -1]
        out.append(torch.stack([msp, knn, vim.predict_features(f)], dim=1).cpu())
    return torch.cat(out).numpy()


pool_scores = {
    "CIFAR-10 (ID)":    score_pool(art["pool_cifar10"],  base_transform),
    "CIFAR-100 (OOD)":  score_pool(art["pool_cifar100"], base_transform),
    "MNIST (OOD)":      score_pool(art["pool_mnist"],    mnist_transform),
}

methods = [("MSP", "high = ID "), ("KNN", "high = OOD"), ("ViM", "high = OOD")]

print("=" * 72)
print(f"MEAN SCORES OVER {sum(len(v) for v in pool_scores.values())} BUNDLED IMAGES")
print("=" * 72)
print(f"{'':<14}" + "".join(f"{name:>20}" for name in pool_scores))
print("-" * 72)
for m, (name, direction) in enumerate(methods):
    row = f"{name + '  ' + direction:<14}"
    row += "".join(f"{v[:, m].mean():>20.3f}" for v in pool_scores.values())
    print(row)
print("=" * 72)

# How often does a random (ID, OOD) pair come out in the wrong order?
print("\nShare of random (ID image, OOD image) pairs ranked the wrong way round:")
print("i.e. how often a single draw makes the method look broken\n")
print(f"{'':<14}" + f"{'vs CIFAR-100':>16}{'vs MNIST':>12}")
print("-" * 44)
id_scores = pool_scores["CIFAR-10 (ID)"]
for m, (name, _) in enumerate(methods):
    sign = -1 if name == "MSP" else 1          # MSP points the other way
    row = f"{name:<14}"
    for other in ("CIFAR-100 (OOD)", "MNIST (OOD)"):
        a = sign * id_scores[:, m]
        b = sign * pool_scores[other][:, m]
        row += f"{100 * (a[:, None] >= b[None, :]).mean():>15.1f}%"
    print(row)

## 6. Reading the results

All three scores come from **the same 128-dimensional feature** of the same image; only the
interpretation differs.

**MSP** looks only at the softmax of the final layer. Softmax normalises over exactly the 10
CIFAR-10 classes, so it has no way to express *"none of these"* — OOD images routinely receive
very high confidence. An MNIST digit scoring 0.99 is not a bug; it is the overconfidence problem
that motivated every method that came after MSP.

**KNN** ignores the classifier head and asks a different question: among the 50,000 training
images, does this one land in a crowded or a sparse region of feature space? The neighbour list
in Cell 5 makes this visible — ID images tend to draw neighbours from a single class, while OOD
images draw a scattered mix.

**ViM** combines both signals: the part of the feature lying **outside** the principal subspace of
the training data (unfamiliar images have a large residual), minus the energy of the logits. That
is why ViM usually beats MSP on near-OOD, where softmax has already saturated.

**Scope of this demo.** It shows how the three methods score individual images, and the pool
averages in Cell 7 confirm the overall direction. It does not establish which method is best:
that requires all 10,000 images per test set and threshold-free metrics such as AUROC and FPR95,
which is what the full notebook `OOD_MSP_KNN_ViM_demo.ipynb` computes.